# Lingolate lexicon build (schema v3)

Runs on a **standard CPU runtime** (no GPU needed). Builds the master lexicon (Open Multilingual Wordnet, TUFS, PanLex, Wikidata Lexemes), adds IPA, frequency ranks and grammar features (Grambank, WALS), exports one pack per language pair, uploads them to a GitHub Release and commits `manifest-v3.json`.

The master DB and source downloads are cached on Google Drive (`MyDrive/lingolate-lexicon-build`), so a rerun only redoes what changed. Set `REBUILD = True` to start from the raw sources.

Needs the Colab secret **`GITHUB_TOKEN`** (or `GH_TOKEN`) with *Contents: read and write* on this repo. The runtime disconnects itself at the end to save compute units.

In [ ]:
VERSION = '2026.11.1'
TAG = f'packs-v{VERSION}'
REBUILD = False   # True: ignore the Drive cache and rebuild the master DB
import os, time
T0 = time.time()
!git clone -q --depth 1 https://github.com/hudaiapa88/lingolate-dict-data.git /content/ldd
%cd /content/ldd
!apt-get -qq install -y espeak-ng > /dev/null
!pip install -q -r scripts/requirements.txt
os.environ['PYTHONIOENCODING'] = 'utf-8'
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/lingolate-lexicon-build'
!mkdir -p {D}
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
# Sources. Restored from Drive when cached; otherwise downloaded:
# Open Multilingual Wordnet, TUFS, PanLex (27 langs), Tatoeba, Wikidata Lexemes, CILI, Grambank, WALS
CACHED = not REBUILD and os.path.exists(f'{D}/lexicon.sqlite3')
!cp -r {D}/sources_cache . 2>/dev/null; true
if CACHED:
    !cp {D}/lexicon.sqlite3 . && cp -r {D}/examples . && ls -la lexicon.sqlite3
else:
    !cd scripts && python download_omw.py > /content/dl_omw.log 2>&1; tail -2 /content/dl_omw.log
    !cd scripts && python download_tufs.py > /content/dl_tufs.log 2>&1; tail -2 /content/dl_tufs.log
    !cd scripts && python download_panlex.py --langs ar,bg,cs,da,de,el,en,es,fi,fr,hi,id,it,ja,ko,lt,nl,no,pl,pt,ru,sv,th,tr,uk,vi,zh > /content/dl_panlex.log 2>&1; tail -3 /content/dl_panlex.log
    !mkdir -p sources_cache/tatoeba && cd sources_cache/tatoeba && for f in sentences_detailed links tags user_languages sentences_CC0; do curl -sSL --retry 5 -o $f.tar.bz2 https://downloads.tatoeba.org/exports/$f.tar.bz2 && bzip2 -t $f.tar.bz2 && echo "$f ok"; done
!python scripts/download_extras.py 2>&1 | tail -3
!rsync -a sources_cache/ {D}/sources_cache/
!du -sh sources_cache/*

In [ ]:
if not CACHED:
    !python scripts/build_tatoeba_examples.py > /content/tatoeba.log 2>&1; tail -2 /content/tatoeba.log
    !python scripts/build_lexicon.py 2>&1 | tail -12
!python scripts/enrich_lexicon.py --steps wikidata,ipa,freq,grammar 2>&1 | tail -60
!cp lexicon.sqlite3 {D}/lexicon.sqlite3 && rsync -a examples/ {D}/examples/

In [ ]:
!python scripts/export_packs.py --version {VERSION} > /content/export.log 2>&1; grep -E 'en-tr|de-tr|ko-tr|exported' /content/export.log
!python scripts/generate_pack_manifest.py --version {VERSION} --tag {TAG}
!mkdir -p {D}/packs && rsync -a packs/ {D}/packs/ && cp manifest-v3.json {D}/

In [ ]:
from google.colab import userdata
def secret(*names):
    for n in names:
        try:
            return userdata.get(n)
        except Exception:
            pass
    raise SystemExit('add a GITHUB_TOKEN secret (Contents: read and write) and enable notebook access')
os.environ['GH_TOKEN'] = secret('GITHUB_TOKEN', 'GH_TOKEN')
!python scripts/upload_packs.py --tag {TAG} --version {VERSION} 2>&1 | tail -4
print(f'total {(time.time() - T0) / 60:.0f} min')

In [ ]:
# Stop the runtime so it does not keep using compute units.
from google.colab import runtime
runtime.unassign()